# Animation Elements for Video 1 of Electric Field Series
## Created by: Antonio Cascio, Blake Head
## Date: 2023-10-02
### Description: This notebook generates individual animation elements for the first video in the Electric Field series.


# Load in Libraries

In [1]:
from manim import *
import manim_physics as mp
import numpy as np
import random

# Define Global Variables to be used throughout video

Here, we define a few objects that are used throughout the video. The functions here are:

- wind_field
    - Creates a vector field that represents the wind field created by the fan
- create_fan
    - Defines and saves parameters to create the fan object.
- create_kite
    - Defines and saves parameters to create the kite object.


In [4]:
def wind_field(point):
    x, y, _ = point
    x -= 1  # Fan origin in x-axis
    y -= 0   # Fan origin in y-axis

    # Compute the magnitude and direction of airflow
    distance = np.sqrt(x**2 + y**2)
    if distance == 0:
        return np.array([0, 0, 0])

    # Magnitude decreases with distance from the fan
    magnitude = 1 / (1 + 0.2 * distance)

    # Flow primarily in x-direction with some spreading in y-direction
    direction = np.array([-1, y / (distance + 0.1), 0])

    # Normalize and scale with magnitude
    return magnitude * direction

def create_fan():
#Create the Fan VGroup
    fan_center = Dot(radius=0.15, color=WHITE)
    fan_body = Circle(radius = 1.2, color=WHITE)
    fan_base = Ellipse(width = 1.8, height= 0.2, color=WHITE).shift(DOWN*3)
    fan_stand = Line(start=fan_body.get_bottom(), end=fan_base.get_top(), color=WHITE)
    
    # Create the fan blades
    blade1 = Polygon(
        ORIGIN, 
        [0.3, 0.3, 0], 
        [0.8, 0, 0], 
        [-0.3, -0.3, 0],
        color=WHITE,
        fill_opacity=0.6
    ).shift(UP * 0.6).rotate(PI / 6)

    # Clone blades and position them
    blade2 = blade1.copy().rotate(PI * 2 / 3, about_point=fan_center.get_center())
    blade3 = blade1.copy().rotate(-PI * 2 / 3, about_point=fan_center.get_center())

    # Group the blades and center
    fan_blades = VGroup(blade1, blade2, blade3)
    fan = VGroup(fan_center, fan_blades, fan_body, fan_base, fan_stand).shift(RIGHT*3)
    
    return fan, fan_blades, fan_body, fan_base, fan_stand, fan_center

def create_kite():
        #Create the kite
    kite_top = Polygon( [-0.6, 0, 0], 
                    [-1, 0.5, 0], 
                    [-0.6, 1, 0], 
                    [-0.2, 0.5, 0], 
                    color=WHITE, 
                    fill_opacity=0.6
                    ).scale(2).shift(LEFT*3)
            
    # Create the string 
    kite_string = VMobject() 
    kite_string.set_points_as_corners( [[-0.6, -1, 0], 
                                        [-0.7, -1.1, 0], 
                                        [-0.5, -1.2, 0], 
                                        [-0.6, -1.3, 0], 
                                        [-0.5, -1.4, 0], 
                                        [-0.6, -1.5, 0]] 
                                    ).scale(3).shift(LEFT*3) 
    
    kite = VGroup(kite_top, kite_string)

    return kite, kite_top, kite_string

# Scene 1: Intro Picture

First we will show the viewer a scene with three point charges, electric force vectors illustrated, as well as Coulomb's law.
This first scene will be to motivate the idea behind electric fields: How do charges know other charges are there?

## (s1p1)

In [83]:
class s1p1(Scene):
    def construct(self):
        # The first part of the video is showcasing the 3 charges

        # Create the charges and set the position
        charge1 = mp.Charge(magnitude=1, point=np.array([-4, -1, 0]))
        charge2 = mp.Charge(magnitude=1, point=np.array([4, 1, 0]))
        charge3 = mp.Charge(magnitude=-1, point=np.array([0, 3, 0]))

        # Add to the scene
        self.play(FadeIn(charge1, charge2, charge3))

        # Create the lines that connect the charges
        line1 = DashedLine(start = charge1.get_center(), end=charge3.get_center(), stroke_width=2)
        line2 = DashedLine(start = charge1.get_center(), end=charge2.get_center(), stroke_width=2)
        line3 = DashedLine(start = charge2.get_center(), end=charge3.get_center(), stroke_width=2)



        ### New force Arrows

        dir_12 = charge1.get_center() - charge2.get_center()
        dir_13 = charge1.get_center() - charge3.get_center()
        dir_23 = charge2.get_center() - charge3.get_center()

        unit_12 = dir_12 / np.linalg.norm(dir_12)
        unit_13 = dir_13 / np.linalg.norm(dir_13)
        unit_23 = dir_23 / np.linalg.norm(dir_23)

        len1 = 1
        len2 = 1.5
        len3 = 2

        arrow_end_12 = charge1.get_center() + unit_12 * len1 # Good
        arrow_end_13 = charge1.get_center() + unit_13 * -len2 # Good

        arrow_end_23 = charge2.get_center() + unit_23 * -len3
        arrow_end_21 = charge2.get_center() + unit_12 * -len1

        arrow_end_31 = charge3.get_center() + unit_13 * len2
        arrow_end_32 = charge3.get_center() + unit_23 * len3

        arrow_12 = Arrow(start=charge1.get_center(), end=arrow_end_12, color=GREEN)
        arrow_13 = Arrow(start=charge1.get_center(), end=arrow_end_13, color=GREEN)

        arrow_23 = Arrow(start=charge2.get_center(), end=arrow_end_23, color=GREEN)
        arrow_21 = Arrow(start=charge2.get_center(), end=arrow_end_21, color=GREEN)

        arrow_31 = Arrow(start=charge3.get_center(), end=arrow_end_31, color=GREEN)
        arrow_32 = Arrow(start=charge3.get_center(), end=arrow_end_32, color=GREEN)

        self.play(FadeIn(arrow_12, arrow_13, arrow_23, arrow_21, arrow_31, arrow_32))

       
        # Equation time
        equation = MathTex(
            "\\vec{F_c}", "=", "k_c", "{q_1", "q_2", "\\over", "r^2}", "\\hat{r}"
        )
        equation.move_to(DOWN*1.5)

        #Add the equation
        self.play(Write(equation))

        #Run the scene for as long as desired
        self.wait(4)

        self.play(FadeIn(line1), FadeIn(line2), FadeIn(line3))

        self.wait(3)

        #Prepare the next scene of the charges floating in space
        self.play(FadeOut(line1,line2,line3
                          ,equation))

        #Scene runtime
        self.wait(4)

        question_mark = Text("?", font_size=144)

        self.play(FadeIn(question_mark))

        self.wait(4)

        #Remove the charges for the next scene and add the kite and fan
        self.play(FadeOut(charge1, charge2, charge3, question_mark))
        

        
# Load the manim extension
%load_ext manim 

# Render the main scene
%manim -qk -v WARNING s1p1

The manim module is not an IPython extension.


Manim Community v0.19.0

# Scene 2: Fan and Kite (pt 1)

In this scene, we add a fan and kite to the scene.  This is a multi-part scene, and is comprised of many moving elements. The primary objects throughout are as follows:

- The fan body
- The rotating fan blades
- The kite
- Foce vector on kite.
- The wind field lines (defined in function above)

In this part 1, the fan and kite are stationary, and the blades are NOT rotating.

Note the first two lines: These are calling our 'create' functions definied up at the top of this file.  
This allows the individual code blocks to be much more compact.  The bracketed numbers says which element of the definition output to assign to our variable. Those elements are:

- [0] = fan
- [1] = fan_blades
- [2] = fan_body
- [3] = fan_base
- [4] = fan_stand
- [5] = fan_center

## (s2p1)

In [256]:
class s2p1(Scene):
        def construct(self):
            fan = create_fan()[0]
            kite = create_kite()[0]

            #Add the Fan and Kite to the scene
            self.play(FadeIn(fan, kite))

            #Running the scene for set time
            self.wait(2)
    
    # Load the manim extension
%load_ext manim 

# Render the main scene
%manim -qk -v WARNING s2p1

The manim module is not an IPython extension.


Manim Community v0.19.0

## (s2p2)

In [257]:
class s2p2(Scene):
    def construct(self):

        # Add in objects present at end of previous scene
        fan = create_fan()[0]
        fan_blades = create_fan()[1]
        fan_body = create_fan()[2]
        fan_base = create_fan()[3]
        fan_stand = create_fan()[4]
        fan_center = create_fan()[5]


        kite_top = create_kite()[1]
        kite_string = create_kite()[2]
        self.add(fan_body, fan_stand, fan_base, fan_center, fan_blades)
        kite = VGroup(kite_top, kite_string)
        self.add(kite)
        # Store the initial position of the kite
        kiteIP = kite.get_center()

    # Next part of the scene is to add the streamlines/wind

    # Create the corner covers (hard coded)
    # There are used to cover the top and bottom right corners to make the field look more correct.

        corner_cover1 = Circle(radius=5, color=BLACK, fill_opacity=1).shift(UP*5+RIGHT*7)
        corner_cover2 = Circle(radius=5, color=BLACK, fill_opacity=1).shift(UP*-5+RIGHT*7)

        
    # Create streamlines from the vector field
    # I define 3 different streamlines that will represent a slow/medium/fast fan speed.
    # I also slightly adjust the virtual_time variable for each, making it a bit larger for faster fields.

    # If things in the animation look a little bad, try changing the 'virtual time' paramenter by a couple.

        stream_lines_slow = StreamLines(
                wind_field,
                stroke_width=2, # Make the lines wider
                max_anchors_per_line=30,
                virtual_time=2 # 
            )

        # This line adds an updater to the fan blades so that they will continue to rotate
        # throughout the animations.  
        # Multiplying the angle by 'speed' allows us to adjust the speed of the fan below.
        fan_blades.add_updater(lambda mob, dt: mob.rotate(angle=speed*dt*PI, 
                                                            about_point=fan_center.get_center())
                                )

    # Here we create the force vector that will illustrate the wind force.
    # Set vector parameters here:
        vec_color = PURE_GREEN
        vec_magnitude = 5
    # Function to compute vector direction and inverse distance magnitude
        def get_vector():
            direction = kite_top.get_center() - fan.get_center()  # Direction from red to blue
            distance = np.linalg.norm(direction)  # Compute distance
                
            if distance == 0:  # Avoid division by zero
                return Vector([0, 0, 0]).set_color(vec_color)

            unit_direction = direction / distance  # Normalize direction
            magnitude = vec_magnitude / distance  # Inverse proportionality (scaled factor for visibility)

            start_point = kite_top.get_center()  # Start at the kite
            end_point = start_point + unit_direction * magnitude  # End point based on direction and magnitude

            return Vector(end_point - start_point).set_color(vec_color).put_start_and_end_on(start_point, end_point)

        # Create initial vector
        vec = get_vector()
            
        # Updater to dynamically adjust vector direction and magnitude
        def update_vector(mob):
            direction = kite_top.get_center() - fan.get_center()  # Direction from fan to kite
            distance = np.linalg.norm(direction)  # Compute distance

            if distance == 0:  # Avoid division by zero
                mob.put_start_and_end_on(kite_top.get_center(), kite_top.get_center())
                return

            unit_direction = direction / distance  # Normalize direction
            magnitude = vec_magnitude / distance  # Inverse proportionality (scaled factor for visibility)

            start_point = kite_top.get_center()  # Start at the kite
            end_point = start_point + unit_direction * magnitude  # End point based on direction and magnitude

            mob.put_start_and_end_on(start_point, end_point)  # Update vector's start and end points

        vec.add_updater(update_vector)

    # Start animations

        # Medium Wind Field

        speed = 0.5 # Change this to make the fan spin faster or slower
        vec_magnitude = 2 # Change this to update the base size of the force vector to account for wind strength.

        # Add in all scene elements
        self.add(stream_lines_slow, corner_cover1, corner_cover2)
        self.add(fan_base,fan_stand)
        self.add(vec)
        # Start the StreamLines animation bit
        # Flow speed determines how quickly the stream lines will move
        stream_lines_slow.start_animation(warm_up=False, flow_speed=.5)
        
        # Wait a moment until moving on
        self.wait(3)

        # Move the kite around
        self.play(kite.animate.move_to(RIGHT*2+DOWN*1), run_time=3)
        self.wait(2)
        self.play(kite.animate.move_to(LEFT*3+UP*0.5), run_time = 3)
        self.wait(2)
        self.play(kite.animate.move_to(kiteIP), run_time = 3)
        self.wait(2)

    # Load the manim extension
%load_ext manim 

# Render the main scene
%manim -qK -v WARNING s2p2

The manim module is not an IPython extension.


Manim Community v0.19.0

## (s2p3)

In [258]:
class s2p3(Scene):
    def construct(self):

        # Add in objects present at end of previous scene
        fan = create_fan()[0]
        fan_blades = create_fan()[1]
        fan_body = create_fan()[2]
        fan_base = create_fan()[3]
        fan_stand = create_fan()[4]
        fan_center = create_fan()[5]


        kite_top = create_kite()[1]
        kite_string = create_kite()[2]
        self.add(fan_body, fan_stand, fan_base, fan_center, fan_blades)
        kite = VGroup(kite_top, kite_string)
        self.add(kite)
        # Store the initial position of the kite
        kiteIP = kite.get_center()

    # Next part of the scene is to add the streamlines/wind

    # Create the corner covers (hard coded)
    # There are used to cover the top and bottom right corners to make the field look more correct.

        corner_cover1 = Circle(radius=5, color=BLACK, fill_opacity=1).shift(UP*5+RIGHT*7)
        corner_cover2 = Circle(radius=5, color=BLACK, fill_opacity=1).shift(UP*-5+RIGHT*7)

        
    # Create streamlines from the vector field
    # I define 3 different streamlines that will represent a slow/medium/fast fan speed.
    # I also slightly adjust the virtual_time variable for each, making it a bit larger for faster fields.

    # If things in the animation look a little bad, try changing the 'virtual time' paramenter by a couple.

        stream_lines_med = StreamLines(
                wind_field,
                stroke_width=2, # Make the lines wider
                max_anchors_per_line=30,
                virtual_time=8 # 
            )

        # This line adds an updater to the fan blades so that they will continue to rotate
        # throughout the animations.  
        # Multiplying the angle by 'speed' allows us to adjust the speed of the fan below.
        fan_blades.add_updater(lambda mob, dt: mob.rotate(angle=speed*dt*PI, 
                                                            about_point=fan_center.get_center())
                                )

    # Here we create the force vector that will illustrate the wind force.
    # Set vector parameters here:
        vec_color = PURE_GREEN
        vec_magnitude = 5
    # Function to compute vector direction and inverse distance magnitude
        def get_vector():
            direction = kite_top.get_center() - fan.get_center()  # Direction from red to blue
            distance = np.linalg.norm(direction)  # Compute distance
                
            if distance == 0:  # Avoid division by zero
                return Vector([0, 0, 0]).set_color(vec_color)

            unit_direction = direction / distance  # Normalize direction
            magnitude = vec_magnitude / distance  # Inverse proportionality (scaled factor for visibility)

            start_point = kite_top.get_center()  # Start at the kite
            end_point = start_point + unit_direction * magnitude  # End point based on direction and magnitude

            return Vector(end_point - start_point).set_color(vec_color).put_start_and_end_on(start_point, end_point)

        # Create initial vector
        vec = get_vector()
            
        # Updater to dynamically adjust vector direction and magnitude
        def update_vector(mob):
            direction = kite_top.get_center() - fan.get_center()  # Direction from fan to kite
            distance = np.linalg.norm(direction)  # Compute distance

            if distance == 0:  # Avoid division by zero
                mob.put_start_and_end_on(kite_top.get_center(), kite_top.get_center())
                return

            unit_direction = direction / distance  # Normalize direction
            magnitude = vec_magnitude / distance  # Inverse proportionality (scaled factor for visibility)

            start_point = kite_top.get_center()  # Start at the kite
            end_point = start_point + unit_direction * magnitude  # End point based on direction and magnitude

            mob.put_start_and_end_on(start_point, end_point)  # Update vector's start and end points

        vec.add_updater(update_vector)

    # Start animations

        # Medium Wind Field

        speed = 1 # Change this to make the fan spin faster or slower
        vec_magnitude = 5 # Change this to update the base size of the force vector to account for wind strength.

        # Add in all scene elements
        self.add(stream_lines_med, corner_cover1, corner_cover2)
        self.add(fan_base,fan_stand)
        self.add(vec)
        # Start the StreamLines animation bit
        # Flow speed determines how quickly the stream lines will move
        stream_lines_med.start_animation(warm_up=False, flow_speed=1.5)
        
        # Wait a moment until moving on
        self.wait(3)

        # Move the kite around
        self.play(kite.animate.move_to(RIGHT*2+DOWN*1), run_time=3)
        self.wait(2)
        self.play(kite.animate.move_to(LEFT*3+UP*0.5), run_time = 3)
        self.wait(2)
        self.play(kite.animate.move_to(kiteIP), run_time = 3)
        self.wait(2)

    # Load the manim extension
%load_ext manim 

# Render the main scene
%manim -qk -v WARNING s2p3

The manim module is not an IPython extension.


Manim Community v0.19.0

## (s2p4)

In [259]:
class s2p4(Scene):
    def construct(self):

        def wait_rotations(scene, rotations, speed):
            seconds = rotations * 2 / speed
            scene.wait(seconds)
            
        # Add in objects present at end of previous scene
        fan = create_fan()[0]
        fan_blades = create_fan()[1]
        fan_body = create_fan()[2]
        fan_base = create_fan()[3]
        fan_stand = create_fan()[4]
        fan_center = create_fan()[5]


        kite_top = create_kite()[1]
        kite_string = create_kite()[2]
        self.add(fan_body, fan_stand, fan_base, fan_center, fan_blades)
        kite = VGroup(kite_top, kite_string)
        self.add(kite)
        # Store the initial position of the kite
        kiteIP = kite.get_center()

    # Next part of the scene is to add the streamlines/wind

    # Create the corner covers (hard coded)
    # There are used to cover the top and bottom right corners to make the field look more correct.

        corner_cover1 = Circle(radius=5, color=BLACK, fill_opacity=1).shift(UP*5+RIGHT*7)
        corner_cover2 = Circle(radius=5, color=BLACK, fill_opacity=1).shift(UP*-5+RIGHT*7)

        
    # Create streamlines from the vector field
    # I define 3 different streamlines that will represent a slow/medium/fast fan speed.
    # I also slightly adjust the virtual_time variable for each, making it a bit larger for faster fields.

    # If things in the animation look a little bad, try changing the 'virtual time' paramenter by a couple.

        stream_lines_slow = StreamLines(
                wind_field,
                stroke_width=2, # Make the lines wider
                max_anchors_per_line=30,
                virtual_time=1 # 
            )

        # This line adds an updater to the fan blades so that they will continue to rotate
        # throughout the animations.  
        # Multiplying the angle by 'speed' allows us to adjust the speed of the fan below.
        fan_blades.add_updater(lambda mob, dt: mob.rotate(angle=speed*dt*PI, 
                                                            about_point=fan_center.get_center())
                                )

    # Here we create the force vector that will illustrate the wind force.
    # Set vector parameters here:
        vec_color = PURE_GREEN
        vec_magnitude = 5
    # Function to compute vector direction and inverse distance magnitude
        def get_vector():
            direction = kite_top.get_center() - fan.get_center()  # Direction from red to blue
            distance = np.linalg.norm(direction)  # Compute distance
                
            if distance == 0:  # Avoid division by zero
                return Vector([0, 0, 0]).set_color(vec_color)

            unit_direction = direction / distance  # Normalize direction
            magnitude = vec_magnitude / distance  # Inverse proportionality (scaled factor for visibility)

            start_point = kite_top.get_center()  # Start at the kite
            end_point = start_point + unit_direction * magnitude  # End point based on direction and magnitude

            return Vector(end_point - start_point).set_color(vec_color).put_start_and_end_on(start_point, end_point)

        # Create initial vector
        vec = get_vector()
            
        # Updater to dynamically adjust vector direction and magnitude
        def update_vector(mob):
            direction = kite_top.get_center() - fan.get_center()  # Direction from fan to kite
            distance = np.linalg.norm(direction)  # Compute distance

            if distance == 0:  # Avoid division by zero
                mob.put_start_and_end_on(kite_top.get_center(), kite_top.get_center())
                return

            unit_direction = direction / distance  # Normalize direction
            magnitude = vec_magnitude / distance  # Inverse proportionality (scaled factor for visibility)

            start_point = kite_top.get_center()  # Start at the kite
            end_point = start_point + unit_direction * magnitude  # End point based on direction and magnitude

            mob.put_start_and_end_on(start_point, end_point)  # Update vector's start and end points

        vec.add_updater(update_vector)

    # Start animations

        # Medium Wind Field

        speed = 0.5 # Change this to make the fan spin faster or slower
        vec_magnitude = 1 # Change this to update the base size of the force vector to account for wind strength.

        # Add in all scene elements
        self.add(stream_lines_slow, corner_cover1, corner_cover2)
        self.add(fan_base,fan_stand)
        self.add(vec)
        # Start the StreamLines animation bit
        # Flow speed determines how quickly the stream lines will move
        stream_lines_slow.start_animation(warm_up=False, flow_speed=0.5)
        
        # Wait a moment until moving on
        self.wait(3)

        # Move the kite around
        self.play(kite.animate.move_to(RIGHT*2+DOWN*1), run_time=3)
        self.wait(2)
        self.play(kite.animate.move_to(LEFT*3+UP*0.5), run_time = 3)
        self.wait(2)
        self.play(kite.animate.move_to(kiteIP), run_time = 3)

        wait_rotations(self, 2, speed)  # waits for 5 fan rotations

        wind_field_txt = Text("Wind Field", font_size=72).to_edge(UP)
        ind_arrow = CurvedArrow(start_point=wind_field_txt.get_bottom(), end_point=ORIGIN, color=WHITE)

        self.play(Write(wind_field_txt))
        self.play(DrawBorderThenFill(ind_arrow))

        wait_rotations(self, 2, speed)  # waits for 5 fan rotations

        self.play(FadeOut(wind_field_txt, ind_arrow, kite, stream_lines_slow, vec))

        wait_rotations(self, 2, speed)  # waits for 2 fan rotations

    # Load the manim extension
%load_ext manim 

# Render the main scene
%manim -qk -v WARNING s2p4

The manim module is not an IPython extension.


Manim Community v0.19.0

# Scene 3

## (s3p1)

In [260]:
class s3p1(Scene):
    def construct(self):

# Add this helper function near the top of your file
        def wait_rotations(scene, rotations, speed):
            seconds = rotations * 2 / speed
            scene.wait(seconds)
        # Add in objects present at end of previous scene
        fan = create_fan()[0]
        fan_blades = create_fan()[1]
        fan_body = create_fan()[2]
        fan_base = create_fan()[3]
        fan_stand = create_fan()[4]
        fan_center = create_fan()[5]

        self.add(fan_body, fan_stand, fan_base, fan_center, fan_blades)

        kite_top = create_kite()[1]
        kite_string = create_kite()[2]
        speed = 1
        # This line adds an updater to the fan blades so that they will continue to rotate
        # throughout the animations.  
        # Multiplying the angle by 'speed' allows us to adjust the speed of the fan below.
        fan_blades.add_updater(lambda mob, dt: mob.rotate(angle=speed*dt*PI, 
                                                            about_point=fan_center.get_center()))
        

        def wind_field(point):
            x, y = point[:2]
            fan_x, fan_y = 3, 0
            dx = x - fan_x
            dy = y - fan_y
            distance = np.sqrt(dx**2 + dy**2)
            fan_radius = 1.2
            y_min = -1  # adjust as needed
            y_max = 1   # adjust as needed

            # Exclude arrows inside the fan body
            if distance < fan_radius:
                return np.array([0, 0])
            # Exclude arrows on the y-axis of the fan body
            if abs(x - fan_x) < 1e-3:
                return np.array([0, 0])


            magnitude = 2 / (1 + 1.2 * distance)
            if x > fan_x:
                # Intake: vectors point toward fan center
                direction = np.array([fan_x - x, fan_y - y])
            else:
                # Output: vectors point away from fan center
                direction = np.array([dx, dy])
            direction = direction / (np.linalg.norm(direction) + 1e-6)
            return magnitude * direction


        vector_field = ArrowVectorField(
            wind_field,
            x_range=[-7, 7, 1],
            y_range=[-4, 4, 1],
            colors=[BLUE, BLUE_B, BLUE_D],
            length_func=lambda norm: norm  # Show true magnitude
        )
        
        wait_rotations(self, 1, speed)  # waits for 5 fan rotations
        self.play(FadeIn(vector_field))

        # Wait a moment until moving on
        wait_rotations(self, 2, speed)  # waits for 5 fan rotations



        


        # Add in the kites


        kite_1 = VGroup(kite_top.copy(), kite_string.copy()).scale(0.4)
        kite_2 = VGroup(kite_top.copy(), kite_string.copy()).scale(0.4)
        kite_3 = VGroup(kite_top.copy(), kite_string.copy()).scale(0.4)
        kite_4 = VGroup(kite_top.copy(), kite_string.copy()).scale(0.4)
        self.play(FadeIn(kite_1, kite_2, kite_3, kite_4))

        self.play(kite_2.animate.move_to(LEFT*5+DOWN*2), kite_3.animate.move_to(RIGHT*5+UP*3), kite_4.animate.move_to(LEFT*1+UP*2))

        wait_rotations(self, 2, speed)  # waits for 5 fan rotations



        self.play(FadeOut(kite_1, kite_2, kite_3, kite_4))

        wait_rotations(self, 2, speed)  # waits for 5 fan rotations

        self.play(FadeIn(kite_1, kite_2, kite_3, kite_4))
        wait_rotations(self, 2, speed)  # waits for 5 fan rotations


        def wind_arrow_for_kite(kite, base_length=1.5):
            pos = kite[0].get_center()
            wind_vec = wind_field(np.array([pos[0], pos[1], 0]))
            # Store the original width for this kite polygon
            original_width = kite[0].get_width()
            arrow = Arrow(
                start=pos,
                end=pos + base_length * np.array([wind_vec[0], wind_vec[1], 0]),
                color=PURE_GREEN,
                buff=0.1,
                stroke_width=6
            )
            def update_arrow(mob):
                pos = kite[0].get_center()
                wind_vec = wind_field(np.array([pos[0], pos[1], 0]))
                # Use the original width for this kite polygon
                scale = kite[0].get_width() / original_width
                mob.put_start_and_end_on(
                    pos,
                    pos + base_length * scale * np.array([wind_vec[0], wind_vec[1], 0])
                )
            arrow.add_updater(update_arrow)
            return arrow
        
        kite_1_arrow = wind_arrow_for_kite(kite_1)
        kite_2_arrow = wind_arrow_for_kite(kite_2)
        kite_3_arrow = wind_arrow_for_kite(kite_3)
        kite_4_arrow = wind_arrow_for_kite(kite_4)

        self.play(
            FadeIn(kite_1_arrow, kite_2_arrow, kite_3_arrow, kite_4_arrow)
        )

        self.play(kite_1.animate.scale(4),
                  kite_2.animate.scale(0.5), 
                  kite_3.animate.scale(2), 
                  kite_4.animate.scale(0.2),
        )

        wait_rotations(self, 5, speed)  # waits for 5 fan rotations

        self.play(FadeOut(kite_2_arrow, kite_3_arrow, kite_4_arrow, kite_2, kite_3, kite_4, vector_field))

        wait_rotations(self, 2, speed)  # waits for 2 fan rotations        wait_rotations(self, 5, speed)  # waits for 5 fan rotations



# Load the manim extension
%load_ext manim

# Render the main scene
%manim -qk -v WARNING s3p1

The manim module is not an IPython extension.


Manim Community v0.19.0

/var/folders/hb/whp5y2w55zx1m7p5g51zdgww0000gn/T/ipykernel_52445/3768121322.py:95: DeprecationWarning: This method is not guaranteed to stay around. Please prefer getting the attribute normally.
  original_width = kite[0].get_width()
Animation 10: FadeIn(Group):   0%|          | 0/60 [00:00<?, ?it/s]/var/folders/hb/whp5y2w55zx1m7p5g51zdgww0000gn/T/ipykernel_52445/3768121322.py:107: DeprecationWarning: This method is not guaranteed to stay around. Please prefer getting the attribute normally.
  scale = kite[0].get_width() / original_width


## (s3p2)

In [261]:
class s3p2(Scene):
    def construct(self):

# Add this helper function near the top of your file
        def wait_rotations(scene, rotations, speed):
            seconds = rotations * 2 / speed
            scene.wait(seconds)
        # Add in objects present at end of previous scene
        fan = create_fan()[0]
        fan_blades = create_fan()[1]
        fan_body = create_fan()[2]
        fan_base = create_fan()[3]
        fan_stand = create_fan()[4]
        fan_center = create_fan()[5]

        self.add(fan_body, fan_stand, fan_base, fan_center, fan_blades)

        kite_top = create_kite()[1]
        kite_string = create_kite()[2]
        speed = 1
        # This line adds an updater to the fan blades so that they will continue to rotate
        # throughout the animations.  
        # Multiplying the angle by 'speed' allows us to adjust the speed of the fan below.
        fan_blades.add_updater(lambda mob, dt: mob.rotate(angle=speed*dt*PI, 
                                                            about_point=fan_center.get_center()))
        

        def wind_field(point):
            x, y = point[:2]
            fan_x, fan_y = 3, 0
            dx = x - fan_x
            dy = y - fan_y
            distance = np.sqrt(dx**2 + dy**2)
            fan_radius = 1.2
            y_min = -1  # adjust as needed
            y_max = 1   # adjust as needed

            # Exclude arrows inside the fan body
            if distance < fan_radius:
                return np.array([0, 0])
            # Exclude arrows on the y-axis of the fan body
            if abs(x - fan_x) < 1e-3:
                return np.array([0, 0])


            magnitude = 2 / (1 + 1.2 * distance)
            if x > fan_x:
                # Intake: vectors point toward fan center
                direction = np.array([fan_x - x, fan_y - y])
            else:
                # Output: vectors point away from fan center
                direction = np.array([dx, dy])
            direction = direction / (np.linalg.norm(direction) + 1e-6)
            return magnitude * direction


        vector_field = ArrowVectorField(
            wind_field,
            x_range=[-7, 7, 1],
            y_range=[-4, 4, 1],
            colors=[BLUE, BLUE_B, BLUE_D],
            length_func=lambda norm: norm  # Show true magnitude
        )
        
        def wind_arrow_for_kite(kite, base_length=1.5):
            pos = kite[0].get_center()
            wind_vec = wind_field(np.array([pos[0], pos[1], 0]))
            original_width = kite[0].get_width()
            scale = kite[0].get_width() / original_width  # This will be 1, but keep for clarity
            arrow = Arrow(
                start=pos,
                end=pos + base_length * scale * np.array([wind_vec[0], wind_vec[1], 0]),  # <-- scale here!
                color=PURE_GREEN,
                buff=0.1,
                stroke_width=6
            )
            def update_arrow(mob):
                pos = kite[0].get_center()
                wind_vec = wind_field(np.array([pos[0], pos[1], 0]))
                scale = kite[0].get_width() / original_width
                mob.put_start_and_end_on(
                    pos,
                    pos + base_length * scale * np.array([wind_vec[0], wind_vec[1], 0])
                )
            arrow.add_updater(update_arrow)
            return arrow
        
        # Scale the kite first
        kite_1 = VGroup(kite_top.copy(), kite_string.copy()).scale(0.4)
        original_pos = kite_1.get_center()

        # Now create the arrow, so original_width matches the scaled kite
        kite_1_arrow = wind_arrow_for_kite(kite_1)

        self.add(kite_1, kite_1_arrow)
        self.play(kite_1.animate.scale(4))
        wait_rotations(self, 2, speed)
        self.play(kite_1.animate.scale(0.6))
        wait_rotations(self, 2, speed)

        self.play(kite_1.animate.move_to(UP*2+RIGHT*2), run_time=3)
        wait_rotations(self, 2, speed)
        self.play(FadeIn(vector_field))
        wait_rotations(self, 1, speed)

        self.play(FadeOut(vector_field))
        self.play(kite_1.animate.move_to(LEFT*4+DOWN*2), run_time=3)
        wait_rotations(self, 1, speed)
        self.play(FadeIn(vector_field))
        wait_rotations(self, 1, speed)


        self.play(FadeOut(vector_field))
        self.play(kite_1.animate.move_to(original_pos), run_time=3)
        wait_rotations(self, 1, speed)
        self.play(FadeIn(vector_field))
        wait_rotations(self, 2, speed)
        self.play(FadeOut(vector_field, kite_1_arrow))
        wait_rotations(self, 1, speed)


# Load the manim extension
%load_ext manim

# Render the main scene
%manim -qk -v WARNING s3p2

The manim module is not an IPython extension.


Manim Community v0.19.0

/var/folders/hb/whp5y2w55zx1m7p5g51zdgww0000gn/T/ipykernel_52445/3728213256.py:68: DeprecationWarning: This method is not guaranteed to stay around. Please prefer getting the attribute normally.
  original_width = kite[0].get_width()
/var/folders/hb/whp5y2w55zx1m7p5g51zdgww0000gn/T/ipykernel_52445/3728213256.py:69: DeprecationWarning: This method is not guaranteed to stay around. Please prefer getting the attribute normally.
  scale = kite[0].get_width() / original_width  # This will be 1, but keep for clarity
Animation 0: _MethodAnimation(VGroup of 2 submobjects):   0%|          | 0/60 [00:00<?, ?it/s]/var/folders/hb/whp5y2w55zx1m7p5g51zdgww0000gn/T/ipykernel_52445/3728213256.py:80: DeprecationWarning: This method is not guaranteed to stay around. Please prefer getting the attribute normally.
  scale = kite[0].get_width() / original_width


# Scene 4

## (s4p1)

In [262]:
class s4p1(Scene):
    def construct(self):

# Add this helper function near the top of your file
        def wait_rotations(scene, rotations, speed):
            seconds = rotations * 2 / speed
            scene.wait(seconds)

        # Add in objects present at end of previous scene
        fan = create_fan()[0]
        fan_blades = create_fan()[1]
        fan_body = create_fan()[2]
        fan_base = create_fan()[3]
        fan_stand = create_fan()[4]
        fan_center = create_fan()[5]

        self.add(fan_body, fan_stand, fan_base, fan_center, fan_blades)

        kite_top = create_kite()[1]
        kite_string = create_kite()[2]
        speed = 1
        # This line adds an updater to the fan blades so that they will continue to rotate
        # throughout the animations.  
        # Multiplying the angle by 'speed' allows us to adjust the speed of the fan below.
        fan_blades.add_updater(lambda mob, dt: mob.rotate(angle=speed*dt*PI, 
                                                            about_point=fan_center.get_center()))
        
        
        # Scale the kite first
        kite_1 = VGroup(kite_top.copy(), kite_string.copy()).scale(0.4).scale(4).scale(0.6)


        self.add(kite_1)
        wait_rotations(self, 1, speed)


    # Create the objects to transition to
        kite_charge = mp.Charge(magnitude=1, point=kite_1.get_center(), add_glow=False)
        fan_charge = mp.Charge(magnitude=5, point=fan_blades.get_center(), add_glow=False)
        
        fan_group = VGroup(fan_body, fan_stand, fan_base, fan_center, fan_blades)
        self.add(fan_group)
        self.remove(fan_body, fan_stand, fan_base, fan_center, fan_blades)

        self.play(Transform(kite_1, kite_charge),Transform(fan_group, fan_charge))
        
        # Create the objects to transition to
        kite_charge_glow = mp.Charge(magnitude=1, point=kite_1.get_center(), add_glow=True)
        fan_charge_glow = mp.Charge(magnitude=5, point=fan_blades.get_center(), add_glow=True)
        self.play(FadeIn(kite_charge_glow, fan_charge_glow), FadeOut(kite_charge, fan_charge))

        self.wait(1)



# Load the manim extension
%load_ext manim

# Render the main scene
%manim -qk -v WARNING s4p1

The manim module is not an IPython extension.


Manim Community v0.19.0

## (s4p2)

In [263]:
class s4p2(Scene):
    def construct(self):

# Add this helper function near the top of your file
        def wait_rotations(scene, rotations, speed):
            seconds = rotations * 2 / speed
            scene.wait(seconds)

        # Add in objects present at end of previous scene
        fan = create_fan()[0]
        fan_blades = create_fan()[1]
        fan_body = create_fan()[2]
        fan_base = create_fan()[3]
        fan_stand = create_fan()[4]
        fan_center = create_fan()[5]


        kite_top = create_kite()[1]
        kite_string = create_kite()[2]
        kite_1 = VGroup(kite_top.copy(), kite_string.copy()).scale(0.4).scale(4).scale(0.6)
        speed = 1
        # This line adds an updater to the fan blades so that they will continue to rotate
        # throughout the animations.  
        # Multiplying the angle by 'speed' allows us to adjust the speed of the fan below.
        fan_blades.add_updater(lambda mob, dt: mob.rotate(angle=speed*dt*PI, 
                                                            about_point=fan_center.get_center()))
        



    # Create the objects to transition to
        kite_charge = mp.Charge(magnitude=1, point=kite_1.get_center())
        fan_charge = mp.Charge(magnitude=5, point=fan_blades.get_center())

        self.add(kite_charge, fan_charge)
        fan_group = VGroup(fan_body, fan_stand, fan_base, fan_center, fan_blades)
        fan_group_1 = fan_group.copy().scale(0.3).move_to(kite_charge.get_center()).shift(DOWN*1)
        fan_group_2 = fan_group.copy().move_to(fan_charge.get_center()).scale(0.3).shift(DOWN*1.2)

        # Remove old updaters
        fan_group_1[4].clear_updaters()
        fan_group_2[4].clear_updaters()

        # Add new updaters referencing each group's center
        speed = 1
        fan_group_1[4].add_updater(lambda mob, dt: mob.rotate(angle=speed*dt*PI, about_point=fan_group_1[3].get_center()))
        fan_group_2[4].add_updater(lambda mob, dt: mob.rotate(angle=speed*dt*PI, about_point=fan_group_2[3].get_center()))

        self.play((FadeIn(fan_group_1), FadeIn(fan_group_2)))
        wait_rotations(self, 2, speed)

        self.play(ShrinkToCenter(fan_group_1))
        wait_rotations(self, 1, speed)
        test_txt = Text("Test", color=YELLOW_C).move_to(kite_charge.get_center()).shift(UP*1)
        self.play(Write(test_txt))
        wait_rotations(self, 2, speed)
        source_txt = Text("Source", color=YELLOW_C).move_to(fan_charge.get_center()).shift(UP*1)
        self.play(Write(source_txt))
        wait_rotations(self, 2, speed)

        self.play(FadeOut(fan_group_2))
        self.wait(1)

        #print("Kite center:", kite_1.get_center())
        #print("Fan blades center:", fan_blades.get_center())




# Load the manim extension
%load_ext manim

# Render the main scene
%manim -qk -v WARNING s4p2

The manim module is not an IPython extension.


Manim Community v0.19.0

## s4p2.5

In [3]:
class s4p25(Scene):
    def construct(self):
        kite_charge = mp.Charge(magnitude=1).move_to(np.array([-3.6,-0.25,0.])) #Pulled these manually from previous cell
        fan_charge = mp.Charge(magnitude=5).move_to(np.array([2.80269238,-0.09084936,0.])) #See print function output

        self.add(kite_charge, fan_charge)

        def get_force_arrow():
            start = kite_charge.get_center()
            end = fan_charge.get_center()
            direction = start - end  # Repulsive: away from fan_charge
            distance = np.linalg.norm(direction)
            if distance == 0:
                return Arrow(start, start, color=GREEN, buff=0.1, stroke_width=6)
            unit_direction = direction / distance
            force_magnitude = 0.5 + 4 / (distance**2)
            arrow_end = start + unit_direction * force_magnitude
            return Arrow(start, arrow_end, color=GREEN, buff=0.1, stroke_width=6)

        force_arrow = get_force_arrow()

        test_txt = Text("Test", color=YELLOW_C).move_to(kite_charge.get_center()).shift(UP*1)
        source_txt = Text("Source", color=YELLOW_C).move_to(fan_charge.get_center()).shift(UP*1)

        def update_arrow(mob):
            start = kite_charge.get_center()
            end = fan_charge.get_center()
            direction = start - end  # Repulsive: away from fan_charge
            distance = np.linalg.norm(direction)
            if distance == 0:
                mob.put_start_and_end_on(start, start)
                return
            unit_direction = direction / distance
            force_magnitude = 0.5 + 4 / (distance**2)
            arrow_end = start + unit_direction * force_magnitude
            mob.put_start_and_end_on(start, arrow_end)

        force_arrow.add_updater(update_arrow)
        self.add(test_txt, source_txt)

        e_field = always_redraw(lambda: mp.ElectricField(fan_charge))
        self.play(FadeIn(e_field, force_arrow))
        self.wait(2)

        big_txt = Text("The Electric Field", color=YELLOW_C).shift(UP*3)
        self.play(Write(big_txt))
        self.wait(2)
        self.play(Unwrite(big_txt))
        self.wait(1)
        

        eforce = Text("Electric Force", color=PURE_GREEN).move_to(kite_charge.get_center()).shift(DOWN*1).scale(0.5)
        self.play(Write(eforce))

        curved_arrow_force = CurvedArrow(
            eforce.get_top(),  # where the arrow starts
            force_arrow.get_center(),  # where the arrow points
            angle=-PI/3,
            color=LIGHT_PINK
        )
        self.play(DrawBorderThenFill(curved_arrow_force))
        self.wait(1)

        efield_txt = Text("Electric Field", color=PURE_GREEN).move_to(e_field.get_center()).shift(UP*1+LEFT*1).scale(0.5)
        self.play(Write(efield_txt))
        self.wait(1)

        random.seed(42)
        curved_arrows = []  # List to hold the arrows

        for _ in range(4):  # 4 more arrows (plus the original makes 5)
            rand_x = random.uniform(-6, 6)
            rand_y = random.uniform(-3.5, 3.5)
            target_point = np.array([rand_x, rand_y, 0])
            curved_arrow = CurvedArrow(
                efield_txt.get_bottom(),
                target_point,
                angle=random.choice([PI/3, -PI/3, PI/4, -PI/4]),
                color=LIGHT_PINK
            )
            curved_arrows.append(curved_arrow)

            # ...existing code...
        curved_arrow1 = CurvedArrow(
            efield_txt.get_bottom(),
            kite_charge.get_center(),
            angle=-PI/3,
            color=LIGHT_PINK
        )

        # Add all arrows at once
        self.play(*[DrawBorderThenFill(arrow) for arrow in curved_arrows], DrawBorderThenFill(curved_arrow1))
        self.wait(1)

        self.play(*[FadeOut(arrow) for arrow in curved_arrows])
        self.wait(2)

        self.play(Circumscribe(eforce))
        self.wait(2)

        self.play(Circumscribe(efield_txt))
        self.wait(2)

        self.play(Circumscribe(test_txt))
        self.wait(2)

        self.play(FadeOut(test_txt, source_txt, eforce, efield_txt, curved_arrow1, curved_arrow_force))
        self.wait(1)

    

# Load the manim extension
%load_ext manim

# Render the main scene
%manim -qk -v WARNING s4p25

The manim module is not an IPython extension.


Manim Community v0.19.0

## (s4p3)

In [4]:
class s4p3(Scene):
    def construct(self):
        kite_charge = mp.Charge(magnitude=1).move_to(np.array([-3.6,-0.25,0.])) #Pulled these manually from previous cell
        fan_charge = mp.Charge(magnitude=5).move_to(np.array([2.80269238,-0.09084936,0.])) #See print function output

        self.add(kite_charge, fan_charge)

        def get_force_arrow():
            start = kite_charge.get_center()
            end = fan_charge.get_center()
            direction = start - end  # Repulsive: away from fan_charge
            distance = np.linalg.norm(direction)
            if distance == 0:
                return Arrow(start, start, color=GREEN, buff=0.1, stroke_width=6)
            unit_direction = direction / distance
            force_magnitude = 0.5 + 4 / (distance**2)
            arrow_end = start + unit_direction * force_magnitude
            return Arrow(start, arrow_end, color=GREEN, buff=0.1, stroke_width=6)

        force_arrow = get_force_arrow()

        def update_arrow(mob):
            start = kite_charge.get_center()
            end = fan_charge.get_center()
            direction = start - end  # Repulsive: away from fan_charge
            distance = np.linalg.norm(direction)
            if distance == 0:
                mob.put_start_and_end_on(start, start)
                return
            unit_direction = direction / distance
            force_magnitude = 0.5 + 4 / (distance**2)
            arrow_end = start + unit_direction * force_magnitude
            mob.put_start_and_end_on(start, arrow_end)

        force_arrow.add_updater(update_arrow)
        self.add(force_arrow)

        e_field = always_redraw(lambda: mp.ElectricField(fan_charge))
        self.add(e_field)
        self.wait(2)

        self.play(kite_charge.animate.move_to(UP*2+LEFT*1), run_time=3)
        self.wait(1)
        self.play(kite_charge.animate.move_to(UP*-1+LEFT*3), run_time=3)
        self.wait(1)
        self.play(kite_charge.animate.move_to(UP*-2+RIGHT*5), run_time=3)
        self.wait(1)
        self.play(kite_charge.animate.move_to(np.array([-3.6,-0.25,0.])), run_time=3)
        self.wait(1)

        self.play(fan_charge.animate.shift(LEFT*2+UP*2), run_time = 3)
        self.wait()
        self.play(fan_charge.animate.shift(LEFT*3+DOWN*2), run_time = 3)
        self.wait()
        self.play(fan_charge.animate.shift(RIGHT*5), run_time = 2)
        self.wait(1)

        # Reverse the force arrow direction in the updater
        def update_arrow(mob):
            start = kite_charge.get_center()
            end = fan_charge.get_center()
            direction = (start - end) if fan_charge.magnitude > 0 else (end - start)
            distance = np.linalg.norm(direction)
            if distance == 0:
                mob.put_start_and_end_on(start, start)
                return
            unit_direction = direction / distance
            force_magnitude = -(0.5 + 4 / (distance**2)) * abs(kite_charge.magnitude)
            arrow_end = start + unit_direction * force_magnitude
            mob.put_start_and_end_on(start, arrow_end)

        force_arrow.clear_updaters()
        force_arrow.add_updater(update_arrow)

        # Animate the flip (for example, after your last wait)
        self.wait(1)
        neg_source = mp.Charge(magnitude=-5).move_to(np.array([2.80269238,-0.09084936,0.]))
        neg_field = mp.ElectricField(neg_source)
        self.play(ReplacementTransform(fan_charge, neg_source), ReplacementTransform(e_field, neg_field))
        self.wait(2)

        self.play(kite_charge.animate.shift(RIGHT*4), run_time = 3)
        self.wait(1)
        self.play(kite_charge.animate.shift(LEFT*4))
        self.wait(1)


# Load the manim extension
%load_ext manim

# Render the main scene
%manim -qk -v WARNING s4p3

The manim module is not an IPython extension.


Manim Community v0.19.0

## (s4p4)

In [2]:
class s4p4p(Scene):
    def construct(self):

        charge1 = mp.Charge(magnitude=1, point=np.array([-4, -1, 0]))
        charge2 = mp.Charge(magnitude=1, point=np.array([4, 1, 0]))
        charge3 = mp.Charge(magnitude=-1, point=np.array([0, 3, 0]))

        ### New force Arrows

        dir_12 = charge1.get_center() - charge2.get_center()
        dir_13 = charge1.get_center() - charge3.get_center()
        dir_23 = charge2.get_center() - charge3.get_center()

        unit_12 = dir_12 / np.linalg.norm(dir_12)
        unit_13 = dir_13 / np.linalg.norm(dir_13)
        unit_23 = dir_23 / np.linalg.norm(dir_23)

        len1 = 1
        len2 = 1.5
        len3 = 2

        arrow_end_12 = charge1.get_center() + unit_12 * len1 # Good
        arrow_end_13 = charge1.get_center() + unit_13 * -len2 # Good

        arrow_end_23 = charge2.get_center() + unit_23 * -len3
        arrow_end_21 = charge2.get_center() + unit_12 * -len1

        arrow_end_31 = charge3.get_center() + unit_13 * len2
        arrow_end_32 = charge3.get_center() + unit_23 * len3

        arrow_12 = Arrow(start=charge1.get_center(), end=arrow_end_12, color=GREEN)
        arrow_13 = Arrow(start=charge1.get_center(), end=arrow_end_13, color=GREEN)

        arrow_23 = Arrow(start=charge2.get_center(), end=arrow_end_23, color=GREEN)
        arrow_21 = Arrow(start=charge2.get_center(), end=arrow_end_21, color=GREEN)

        arrow_31 = Arrow(start=charge3.get_center(), end=arrow_end_31, color=GREEN)
        arrow_32 = Arrow(start=charge3.get_center(), end=arrow_end_32, color=GREEN)

        efield_1 = mp.ElectricField(charge1)
        efield_2 = mp.ElectricField(charge2)
        efield_3 = mp.ElectricField(charge3)

        e_field = mp.ElectricField(charge1, charge2, charge3)

        self.play(FadeIn(charge1))
        self.play(FadeIn(charge2))
        self.play(FadeIn(charge3))
        self.wait(1)

        self.play(FadeIn(efield_1))
        self.wait(1)
        self.play(GrowArrow(arrow_21), GrowArrow(arrow_31))


        self.play(FadeOut(efield_1), FadeIn(efield_2))
        self.wait(1)
        self.play(GrowArrow(arrow_32), GrowArrow(arrow_12))

        self.play(FadeOut(efield_2), FadeIn(efield_3))
        self.wait(1)
        self.play(GrowArrow(arrow_13), GrowArrow(arrow_23))
        self.wait(2)



# Load the manim extension
%load_ext manim

# Render the main scene
%manim -qk -v WARNING s4p4p

The manim module is not an IPython extension.


Manim Community v0.19.0

# Scene 5

In [63]:
class s5p1(Scene):
    def construct(self):
        # Create the charge and its field
        charge_pos = mp.Charge(magnitude=2)
        field_pos = mp.ElectricField(charge_pos)


        self.play(FadeIn(charge_pos))
        self.wait(1)
        self.play(FadeIn(field_pos))
        self.wait(1)

        # Parameters for radial lines
        num_lines = 18  # 16 looks good
        line_length = 8  # Adjust as needed to go beyond scene edge

        # Center of charge
        center = charge_pos.get_center()

        # Create and add lines
        lines = VGroup()
        for i in range(num_lines):
            angle = i * TAU / num_lines  # TAU = 2*pi
            direction = np.array([np.cos(angle), np.sin(angle), 0])
            start = center
            end = center + direction * line_length
            line = Line(start, end, color=YELLOW, stroke_width=2)
            arrow = Arrow(start, end/(0.35*line_length), color=YELLOW, buff=0, stroke_width=1)
            lines.add(line)
            lines.add(arrow)

        self.play(FadeIn(lines), FadeOut(field_pos))
        self.wait(1)

        angle = 40*DEGREES  # 45 degrees
        direction = np.array([np.cos(angle), np.sin(angle), 0])
        start_pos = center + direction * 3  # 3 units from center
        end_pos = center + direction * 8    # move to 8 units from center

        test = mp.Charge(magnitude=1, point=start_pos, add_glow=False)
        self.play(FadeIn(test))
        self.play(test.animate.move_to(end_pos), run_time=3)

        box = Square(side_length=1, color=WHITE)
        self.play(Create(box))
        self.wait(1)
        self.play(box.animate.shift(RIGHT*3), run_time=4)
        self.wait(3)
        self.play(box.animate.shift(RIGHT*1.5+UP*2.5), run_time=2)
        self.wait(2) 

        num_lines = 72  # 16 looks good
        lines2 = VGroup()

        for i in range(num_lines):
            angle = i * TAU / num_lines  # TAU = 2*pi
            direction = np.array([np.cos(angle), np.sin(angle), 0])
            start = center
            end = center + direction * line_length
            line = Line(start, end, color=YELLOW, stroke_width=2)
            arrow = Arrow(start, end/(0.35*line_length), color=YELLOW, buff=0, stroke_width=1)
            lines2.add(line)
            lines2.add(arrow)
        self.play(FadeIn(lines2))
        self.wait(2)
        self.play(FadeOut(lines2))
        self.wait(1)

        self.play(Uncreate(box))
        self.play(FadeOut(lines), FadeIn(field_pos))
        self.wait(1)

        # Use a box to illustrate line density



        # Go to negative stuff
        charge_neg = mp.Charge(magnitude=-2)
        field_neg = mp.ElectricField(charge_neg)


        self.play(FadeOut(charge_pos), FadeIn(charge_neg), ReplacementTransform(field_pos, field_neg))
              # Create and add lines
        lines = VGroup()
        num_lines = 18  # 16 looks good

        for i in range(num_lines):
            angle = i * TAU / num_lines  # TAU = 2*pi
            direction = np.array([np.cos(angle), np.sin(angle), 0])
            end = center
            start = center + direction * line_length
            line = Line(start, end, color=YELLOW, stroke_width=2)
            arrow = Arrow(start, end/(0.35*line_length), color=YELLOW, buff=2, stroke_width=1)
            lines.add(line)
            lines.add(arrow)

        self.wait(2)
        self.play(FadeIn(lines), FadeOut(field_neg))
        self.wait(2)

        angle = 40*DEGREES  # 45 degrees
        direction = np.array([np.cos(angle), np.sin(angle), 0])
        start_pos = center + direction * 3  # 3 units from center
        end_pos = center     # move to 8 units from center

        test = mp.Charge(magnitude=1, point=start_pos, add_glow=False)
        self.play(FadeIn(test))
        self.play(test.animate.move_to(end_pos), run_time=3)

        self.play(FadeOut(test))
        self.wait(2)

        self.play(FadeOut(lines), FadeOut(charge_neg))
        self.wait(1)


# Load the manim extension
%load_ext manim

# Render the main scene
%manim -qk -v WARNING s5p1

The manim module is not an IPython extension.


Manim Community v0.19.0

In [13]:
class s5p2(Scene):
    def construct(self):

        text = Text("Charges create Electric Fields", color=YELLOW_C).shift(DOWN*1)
        buffer = 0.5  # Adjust as needed
        max_width = config.frame_width - 2 * buffer
        if text.width > max_width:
            text.scale(max_width / text.width)
        self.play(Write(text))
        self.wait(2)
        self.play(Unwrite(text))
        self.wait(1)

        text = Text("The strength of a field is determined \n by the source of the field", color=YELLOW_C).shift(DOWN*1)
        buffer = 0.5  # Adjust as needed
        max_width = config.frame_width - 2 * buffer
        if text.width > max_width:
            text.scale(max_width / text.width)
        self.play(Write(text))
        self.wait(2)
        self.play(Unwrite(text))
        self.wait(1)

        text = Text("The shape of the field is determined \n by the shape of the charge", color=YELLOW_C).shift(DOWN*1)
        buffer = 0.5  # Adjust as needed
        max_width = config.frame_width - 2 * buffer
        if text.width > max_width:
            text.scale(max_width / text.width)
        self.play(Write(text))
        self.wait(2)
        self.play(Unwrite(text))
        self.wait(1)

        text = Text("Field lines point away from positive charges \n and toward negative charges", color=YELLOW_C).shift(DOWN*1)
        buffer = 0.5  # Adjust as needed
        max_width = config.frame_width - 2 * buffer
        if text.width > max_width:
            text.scale(max_width / text.width)
        self.play(Write(text))
        self.wait(2)
        self.play(Unwrite(text))
        self.wait(1)

        text = Text("The density of field lines indicates \n the strength of the electric field", color=YELLOW_C).shift(DOWN*1)
        buffer = 0.5  # Adjust as needed
        max_width = config.frame_width - 2 * buffer
        if text.width > max_width:
            text.scale(max_width / text.width)
        self.play(Write(text))
        self.wait(2)
        self.play(Unwrite(text))
        self.wait(1)

# Load the manim extension
%load_ext manim

# Render the main scene
%manim -qk -v WARNING s5p2

The manim module is not an IPython extension.


Manim Community v0.19.0